In [24]:
from pynq import Overlay
ol_1 = Overlay("nodma_1-Copy1.bit")

In [25]:
import numpy as np
samples = 1250000

import time
my_list=[]
ol_1.reg_bank_0.write(0x40,samples)
ol_1.reg_bank_0.write(0x44,1)
ol_1.reg_bank_0.write(0x44,0)
ol_1.double_derivative_top_0.write(0x00, 1)
print(ol_1.double_derivative_top_0.read(0x00))

while(1):
    if ol_1.double_derivative_top_0.read(0x00) & 0x2:
        print(ol_1.double_derivative_top_0.read(0x00))
        num_1    = ol_1.double_derivative_top_0.read(0x10)
        num_2    = ol_1.double_derivative_top_0.read(0x14)
        den_1    = ol_1.double_derivative_top_0.read(0x28)
        den_2    = ol_1.double_derivative_top_0.read(0x2c)
        break

num = np.int64((num_2 << 32) | (num_1 & 0xFFFFFFFF))
den = np.int64((den_2 << 32) | (den_1 & 0xFFFFFFFF))

import math
dt = 1.0 / 1e6
A  = num / den if den != 0 else 0
#f_est = math.sqrt(A) / (4 * math.pi * dt) if A > 0 else 0
f_est = np.arcsin(np.sqrt(A / 4)) / (np.pi*2 * dt)
print(f_est)

1
4
50000.97152522857


In [26]:
ol = Overlay("design_1_wrapper.bit")

In [27]:
import ipywidgets as widgets
from IPython.display import display

# Your frequency calculations
f_fpga = 125e6
phase = f_fpga / f_est

slider = widgets.FloatSlider(
    min=0,
    max=2,
    step=0.01,
    value=0,
    readout=False,
    continuous_update=True
)

ui = widgets.HBox([
    widgets.Label("0"),
    slider,
    widgets.Label("2π")
])

display(ui)

def update_phase(change):
    x = change["new"]      # 0 to 2

    delay = int((x * phase)/2)

    ol.axi_gpio_0.channel2.write(delay, 0xFFFF)

    print(f"\rx={x:.3f}, delay={delay}", end="")

slider.observe(update_phase, names="value")

x=0.000, delay=0429